# ShopNest Store — Exploratory Data Analysis (EDA)

**Dataset:** ShopNest E-commerce Store (based on Brazilian e-commerce data)

**Tools Used:** Python, Pandas, Matplotlib, Seaborn

**Author:** Giridhar Namballa

---

## What This Project Is About

ShopNest is an online store with orders, products, customers, sellers, payments, and reviews.

In this project, I explored the data to answer simple but important business questions like:
- How many orders are placed every month?
- Which product categories sell the most?
- How do customers pay?
- Are orders being delivered on time?
- Which states have the most customers?


## Dataset Overview

The dataset has **9 CSV files** that are linked together:

| File | What it contains |
|------|------------------|
| orders.csv | All order details with timestamps and status |
| customers.csv | Customer location info |
| order_items.csv | Product and price info per order |
| payments.csv | Payment type and amount |
| reviews.csv | Customer ratings and comments |
| products.csv | Product details and category |
| sellers.csv | Seller location info |
| category_translation.csv | Portuguese to English category names |


## Step 1 — Import Libraries

In [ ]:
# Importing all the libraries we need
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set a clean visual style for all charts
sns.set_theme(style='whitegrid')

# Color palette we will use throughout
COLORS = ['#2196F3','#FF9800','#4CAF50','#F44336','#9C27B0',
          '#00BCD4','#FF5722','#3F51B5','#8BC34A','#E91E63']

print('Libraries imported successfully!')

## Step 2 — Load All CSV Files

In [ ]:
# Loading all 8 CSV files into separate DataFrames
orders    = pd.read_csv('data/orders.csv')
customers = pd.read_csv('data/customers.csv')
items     = pd.read_csv('data/order_items.csv')
payments  = pd.read_csv('data/payments.csv')
reviews   = pd.read_csv('data/reviews.csv')
products  = pd.read_csv('data/products.csv')
sellers   = pd.read_csv('data/sellers.csv')
cats      = pd.read_csv('data/category_translation.csv')

print('All files loaded!')
print(f'Orders: {orders.shape[0]:,} rows')
print(f'Customers: {customers.shape[0]:,} rows')
print(f'Order Items: {items.shape[0]:,} rows')
print(f'Payments: {payments.shape[0]:,} rows')
print(f'Reviews: {reviews.shape[0]:,} rows')
print(f'Products: {products.shape[0]:,} rows')
print(f'Sellers: {sellers.shape[0]:,} rows')

## Step 3 — First Look at the Data

In [ ]:
# Let's see what the orders table looks like
orders.head()

In [ ]:
# Check columns and data types
orders.info()

In [ ]:
# Check how many missing values we have
print('Missing values in orders table:')
print(orders.isnull().sum())

## Step 4 — Clean the Data

The date columns are stored as text (string). We need to convert them to proper date format.
Also, we merge the product category translation so we get English category names.


In [ ]:
# Convert date columns from text to datetime format
date_cols = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], dayfirst=True, errors='coerce')

# Extract useful time parts from order date
orders['year']       = orders['order_purchase_timestamp'].dt.year
orders['month']      = orders['order_purchase_timestamp'].dt.month
orders['month_name'] = orders['order_purchase_timestamp'].dt.strftime('%b')
orders['day_of_week']= orders['order_purchase_timestamp'].dt.day_name()

# Merge category translation to get English names
products = products.merge(cats, on='product_category_name', how='left')

print('Data cleaning done!')
print('Date columns converted. Year/Month/Day columns added.')

## Step 5 — Basic Stats and Summary Numbers

In [ ]:
# Total orders, revenue, and average order value
total_orders  = orders['order_id'].nunique()
total_revenue = payments['payment_value'].sum()
avg_order_val = payments.groupby('order_id')['payment_value'].sum().mean()
avg_review    = reviews['review_score'].mean()
total_sellers = sellers['seller_id'].nunique()
total_products= products['product_id'].nunique()

print('='*45)
print(f'  Total Orders       : {total_orders:>10,}')
print(f'  Total Revenue      : R$ {total_revenue:>10,.0f}')
print(f'  Avg Order Value    : R$ {avg_order_val:>10.2f}')
print(f'  Avg Review Score   : {avg_review:>10.2f} / 5')
print(f'  Total Sellers      : {total_sellers:>10,}')
print(f'  Total Products     : {total_products:>10,}')
print('='*45)

## Chart 1 — Monthly Orders Trend

**Question:** How did order volume change month by month?

This tells us if the business is growing or declining over time.


In [ ]:
# Filter only delivered orders and group by month
monthly = orders[orders['order_status'] == 'delivered'].groupby(
    orders['order_purchase_timestamp'].dt.to_period('M')
).size().reset_index()
monthly.columns = ['month', 'order_count']
monthly['month'] = monthly['month'].astype(str)
monthly = monthly[monthly['month'] >= '2017-01']

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly['month'], monthly['order_count'],
        color='#2196F3', linewidth=2.5, marker='o', markersize=5)
ax.fill_between(range(len(monthly)), monthly['order_count'], alpha=0.15, color='#2196F3')
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['month'], rotation=45, ha='right', fontsize=9)
ax.set_title('Monthly Order Count Trend', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Month')
ax.set_ylabel('Number of Orders')
plt.tight_layout()
plt.savefig('charts/01_monthly_orders_trend.png', dpi=130)
plt.show()
print(f'Peak month: {monthly.loc[monthly.order_count.idxmax(), "month"]} '
      f'({monthly.order_count.max():,} orders)')

## Chart 2 — Order Status Distribution

**Question:** What is the status of all orders — delivered, cancelled, etc.?


In [ ]:
status_counts = orders['order_status'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(status_counts.index, status_counts.values,
              color=COLORS[:len(status_counts)], edgecolor='white')
for bar, val in zip(bars, status_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Order Status Distribution', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Order Status')
ax.set_ylabel('Number of Orders')
plt.tight_layout()
plt.savefig('charts/02_order_status.png', dpi=130)
plt.show()

delivered_pct = status_counts['delivered'] / status_counts.sum() * 100
print(f'Delivered orders: {delivered_pct:.1f}% of all orders')

## Chart 3 — Top 10 Product Categories

**Question:** Which product categories get the most orders?


In [ ]:
# Merge items with products to get category names
prod_orders = items.merge(
    products[['product_id', 'product_category_name_english']],
    on='product_id', how='left'
)
top_cats = prod_orders['product_category_name_english'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_cats.index[::-1], top_cats.values[::-1],
               color='#2196F3', edgecolor='white')
for bar, val in zip(bars, top_cats.values[::-1]):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
ax.set_title('Top 10 Product Categories by Orders', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Number of Orders')
plt.tight_layout()
plt.savefig('charts/03_top_categories.png', dpi=130)
plt.show()

## Chart 4 — Payment Method Distribution

**Question:** How do customers prefer to pay?


In [ ]:
pay_counts = payments[payments['payment_type'] != 'not_defined']['payment_type'].value_counts()

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    pay_counts.values,
    labels=pay_counts.index,
    autopct='%1.1f%%',
    colors=COLORS[:len(pay_counts)],
    startangle=140
)
for text in autotexts:
    text.set_fontsize(11)
    text.set_fontweight('bold')
ax.set_title('Payment Method Distribution', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('charts/04_payment_methods.png', dpi=130)
plt.show()

print('Most used payment method:', pay_counts.index[0],
      f'({pay_counts.values[0]/pay_counts.sum()*100:.1f}%)')

## Chart 5 — Customer Review Score Distribution

**Question:** What ratings do customers give — mostly positive or negative?


In [ ]:
score_counts = reviews['review_score'].value_counts().sort_index()
bar_colors = ['#F44336', '#FF9800', '#FFC107', '#8BC34A', '#4CAF50']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(score_counts.index, score_counts.values,
              color=bar_colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, score_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Customer Review Score Distribution', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Review Score (1 = Worst, 5 = Best)')
ax.set_ylabel('Number of Reviews')
ax.set_xticks([1, 2, 3, 4, 5])
plt.tight_layout()
plt.savefig('charts/05_review_scores.png', dpi=130)
plt.show()

five_star_pct = score_counts[5] / score_counts.sum() * 100
print(f'5-star reviews: {five_star_pct:.1f}% of all reviews')
print(f'Average review score: {reviews["review_score"].mean():.2f}')

## Chart 6 — Revenue by Payment Method

**Question:** Which payment method brings in the most revenue?


In [ ]:
pay_rev = payments[payments['payment_type'] != 'not_defined'].groupby(
    'payment_type')['payment_value'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(pay_rev.index, pay_rev.values,
              color=COLORS[:len(pay_rev)], edgecolor='white')
for bar, val in zip(bars, pay_rev.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50000,
            f'R${val/1e6:.1f}M', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Total Revenue by Payment Method', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Payment Type')
ax.set_ylabel('Total Revenue (R$)')
plt.tight_layout()
plt.savefig('charts/06_revenue_by_payment.png', dpi=130)
plt.show()

## Chart 7 — Delivery Time Distribution

**Question:** How many days does it take to deliver an order?

Delivery time = date order was delivered minus date order was placed.


In [ ]:
# Filter only delivered orders
delivered = orders[orders['order_status'] == 'delivered'].copy()

# Calculate delivery days
delivered['delivery_days'] = (
    delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']
).dt.days

# Remove extreme outliers for clean chart
delivered = delivered[delivered['delivery_days'].between(0, 60)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(delivered['delivery_days'], bins=30,
        color='#2196F3', edgecolor='white', linewidth=0.5)
avg_days = delivered['delivery_days'].mean()
ax.axvline(avg_days, color='#F44336', linewidth=2,
           linestyle='--', label=f'Average: {avg_days:.1f} days')
ax.set_title('Delivery Time Distribution (Days)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Number of Days to Deliver')
ax.set_ylabel('Number of Orders')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('charts/07_delivery_time.png', dpi=130)
plt.show()

print(f'Average delivery time: {avg_days:.1f} days')
print(f'Fastest delivery: {delivered["delivery_days"].min()} days')
print(f'Longest delivery: {delivered["delivery_days"].max()} days')

## Chart 8 — Top 10 States by Number of Customers

**Question:** Which states have the most customers?


In [ ]:
state_counts = customers['customer_state'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(state_counts.index, state_counts.values,
              color='#4CAF50', edgecolor='white')
for bar, val in zip(bars, state_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Top 10 States by Number of Customers', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('State')
ax.set_ylabel('Number of Customers')
plt.tight_layout()
plt.savefig('charts/08_customers_by_state.png', dpi=130)
plt.show()

top_state = state_counts.index[0]
print(f'Top state: {top_state} with {state_counts.values[0]:,} customers')

## Chart 9 — Price Distribution Across Top 5 Categories

**Question:** What is the price range in each top product category?

A box plot shows us the minimum, maximum, and average price in each category.


In [ ]:
top5_cats = prod_orders['product_category_name_english'].value_counts().head(5).index
top5_data = prod_orders[prod_orders['product_category_name_english'].isin(top5_cats)]
top5_data = top5_data[top5_data['price'] < 1000]  # remove extreme outliers

fig, ax = plt.subplots(figsize=(11, 6))
top5_data.boxplot(
    column='price',
    by='product_category_name_english',
    ax=ax, patch_artist=True,
    boxprops=dict(facecolor='#2196F3', alpha=0.6),
    medianprops=dict(color='#F44336', linewidth=2)
)
ax.set_title('Price Distribution — Top 5 Categories', fontsize=14, fontweight='bold', pad=15)
plt.suptitle('')
ax.set_xlabel('Product Category')
ax.set_ylabel('Price (R$)')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('charts/09_price_distribution.png', dpi=130)
plt.show()

## Chart 10 — Orders by Day of Week

**Question:** On which days of the week do customers order the most?


In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_counts = orders.groupby('day_of_week').size().reindex(day_order)

fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = ['#2196F3' if d not in ['Saturday','Sunday'] else '#FF9800'
              for d in dow_counts.index]
bars = ax.bar(dow_counts.index, dow_counts.values,
              color=bar_colors, edgecolor='white')
for bar, val in zip(bars, dow_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 150,
            f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Orders by Day of Week\n(Blue = Weekday, Orange = Weekend)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Day of Week')
ax.set_ylabel('Number of Orders')
plt.tight_layout()
plt.savefig('charts/10_orders_by_day.png', dpi=130)
plt.show()

busiest_day = dow_counts.idxmax()
print(f'Busiest day: {busiest_day} with {dow_counts.max():,} orders')

## Summary — Key Findings

After analysing the ShopNest dataset, here are the main findings:

1. **Order Growth** — Orders grew steadily from 2017 to mid-2018, with a clear peak.
2. **Delivery Rate** — Over 95% of orders are successfully delivered.
3. **Top Categories** — Bed/Bath/Table, Sports, and Furniture are the most ordered categories.
4. **Payment** — Credit card is the most preferred payment method (73%+).
5. **Customer Ratings** — Most customers give 5-star ratings (positive experience).
6. **Delivery Time** — Average delivery takes around 12 days.
7. **Geography** — São Paulo (SP) has the highest number of customers by far.
8. **Weekday Orders** — Customers order more on weekdays than weekends.

---
**Author:** Giridhar Namballa | Civil Engineer → Data Analyst

[GitHub](https://github.com/ashkinzz1729) | [LinkedIn](https://linkedin.com/in/giridharnamballa-a9333a7a)
